# Net-Payout-Based Duration

## Step 0: Setup, Imports, Pfade, Session


In [24]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path
import time
import random
import statsmodels.api as sm

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="lseg"
)

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

## Step 1: Universum laden

In [25]:
# Load quarterly EURO500 net-payout panel
# This file is now the single source for Step 3 inputs.
df_const = load_parquet("euro500_netpayout").copy()

# Ensure key and time index for Step 3
if "firm_id" not in df_const.columns:
    raise KeyError("euro500_netpayout must contain firm_id")

if "year" not in df_const.columns:
    date_candidates = [c for c in ["date_end", "date", "asof_date", "effective_date", "formation_date"] if c in df_const.columns]
    if not date_candidates:
        raise KeyError("euro500_netpayout needs either year or a date column to derive year")
    dt = pd.to_datetime(df_const[date_candidates[0]], errors="coerce")
    df_const["year"] = dt.dt.year

if "date_end" not in df_const.columns:
    if "date" in df_const.columns:
        df_const["date_end"] = pd.to_datetime(df_const["date"], errors="coerce")
    else:
        df_const["date_end"] = pd.NaT

panel_step2 = df_const.copy()

# Lightweight universe table by firm_id (metadata kept for downstream merge)
df_firm = (
    df_const[[c for c in ["firm_id", "name", "hq_country", "hq_code", "trbc_sector", "trbc_sector_code"] if c in df_const.columns]]
    .dropna(subset=["firm_id"])
    .drop_duplicates(subset=["firm_id"])
    .sort_values("firm_id")
    .reset_index(drop=True)
)

firm_list = df_firm["firm_id"].astype(str).tolist()
print(f"Loaded EURO500 net-payout universe with {len(firm_list)} unique firm_id.")


Loaded EURO500 net-payout universe with 1166 unique firm_id.


## Step 2:  Firm-year Masterpanel (ME, BE, A, Sales, NI, GP, Debt, Div, Buyback)


In [26]:
def build_masterpanel_firmyear(panel_step2):
    """
    Input: panel_step2 from euro500_netpayout
      expected key cols:
        firm_id, year (or a date column that can be converted to year)

    Output: masterpanel (firm_id, year) with:
      - harmonized accounting/value columns
      - payouts: PO = dividends + buybacks
      - lags: *_lag1
      - deltas/averages: dBE, avgBE, avgAssets
    """
    df = panel_step2.copy()

    # ---- key columns ----
    if "firm_id" not in df.columns:
        raise ValueError("panel_step2 must contain firm_id")

    if "year" not in df.columns:
        date_candidates = [c for c in ["date_end", "date", "asof_date", "effective_date", "formation_date"] if c in df.columns]
        if not date_candidates:
            raise ValueError("panel_step2 must contain year or a usable date column")
        df["year"] = pd.to_datetime(df[date_candidates[0]], errors="coerce").dt.year

    df["firm_id"] = df["firm_id"].astype(str)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["firm_id", "year"]).copy()
    df["year"] = df["year"].astype(int)

    # ---- harmonize column names from euro500_netpayout ----
    rename_map = {
        "mcap_eur": "ME",
        "Sales": "sales",
        "NetIncome": "net_income",
        "GrossProfit": "gross_profit",
        "Dividends": "dividends",
        "Buybacks": "buybacks",
    }
    existing_renames = {k: v for k, v in rename_map.items() if k in df.columns}
    if existing_renames:
        df = df.rename(columns=existing_renames)

    if "date_end" not in df.columns and "date" in df.columns:
        df["date_end"] = pd.to_datetime(df["date"], errors="coerce")

    # ---- keep only relevant columns if they exist ----
    cols = [
        "firm_id", "year", "date_end",
        "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
        "dividends", "buybacks"
    ]
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    # ---- enforce numeric on value columns ----
    value_cols = [c for c in ["ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt", "dividends", "buybacks"] if c in df.columns]
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # ---- deduplicate: one row per (firm_id, year) ----
    sort_cols = ["firm_id", "year"] + (["date_end"] if "date_end" in df.columns else [])
    df = df.sort_values(sort_cols).groupby(["firm_id", "year"], as_index=False).last()

    # ---- payout components: treat missing as zero ----
    if "dividends" in df.columns:
        df["dividends"] = df["dividends"].fillna(0.0)
    else:
        df["dividends"] = 0.0

    if "buybacks" in df.columns:
        df["buybacks"] = df["buybacks"].fillna(0.0)
    else:
        df["buybacks"] = 0.0

    # ---- total payouts ----
    df["PO"] = df["dividends"] + df["buybacks"]

    # ---- sort + lag variables (t-1) ----
    df = df.sort_values(["firm_id", "year"]).reset_index(drop=True)

    lag_vars = [c for c in ["ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt", "dividends", "buybacks", "PO"] if c in df.columns]
    for c in lag_vars:
        df[f"{c}_lag1"] = df.groupby("firm_id")[c].shift(1)

    # ---- deltas and averages used in profitability states ----
    df["dBE"] = df["BE"] - df["BE_lag1"]
    df["avgBE"] = 0.5 * (df["BE"] + df["BE_lag1"])
    df["avgAssets"] = 0.5 * (df["assets"] + df["assets_lag1"])

    # ---- basic sanity ----
    df.loc[df["ME"] <= 0, "ME"] = pd.NA
    df.loc[df["BE"] <= 0, "BE"] = pd.NA
    df.loc[df["assets"] <= 0, "assets"] = pd.NA
    df.loc[df["sales"] <= 0, "sales"] = pd.NA
    df.loc[df["debt"] < 0, "debt"] = pd.NA

    return df


In [27]:
master = build_masterpanel_firmyear(panel_step2)

master.head()

,firm_id,year,date_end,ME,BE,assets,sales,net_income,gross_profit,debt,...,sales_lag1,net_income_lag1,gross_profit_lag1,debt_lag1,dividends_lag1,buybacks_lag1,PO_lag1,dBE,avgBE,avgAssets
0,FIRM0000001,1998,1998-12-31,142904483.073431,9.267012e+07,2.019414e+08,2.960978e+08,1.538477e+07,1.450279e+08,5.410082e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FIRM0000001,1999,1999-12-31,111992999.999999,1.042371e+08,2.644079e+08,3.053139e+08,1.530706e+07,1.489179e+08,1.013191e+08,...,2.960978e+08,1.538477e+07,1.450279e+08,5.410082e+07,1.002132e+07,0.0,1.002132e+07,1.156696e+07,9.845360e+07,2.331747e+08
2,FIRM0000001,2000,2000-12-29,204792999.999999,1.087722e+08,2.651038e+08,3.053139e+08,1.530706e+07,1.489179e+08,9.915228e+07,...,3.053139e+08,1.530706e+07,1.489179e+08,1.013191e+08,1.002132e+07,0.0,1.002132e+07,4.535159e+06,1.065047e+08,2.647559e+08
3,FIRM0000001,2001,2001-12-31,162137749.999999,1.084580e+08,2.591870e+08,3.797310e+08,1.285700e+07,1.837410e+08,9.405300e+07,...,3.053139e+08,1.530706e+07,1.489179e+08,9.915228e+07,1.104697e+07,0.0,1.104697e+07,-3.142348e+05,1.086151e+08,2.621454e+08
4,FIRM0000001,2002,2002-12-31,145914500.0,1.072090e+08,2.512660e+08,3.504880e+08,1.212600e+07,1.733020e+08,8.685200e+07,...,3.797310e+08,1.285700e+07,1.837410e+08,9.405300e+07,1.188800e+07,0.0,1.188800e+07,-1.249000e+06,1.078335e+08,2.552265e+08


## Step 3: Construction of Firm-Level State Variables



In this step, we construct the **firm-level state variables** that summarize valuation, growth, profitability, and capital structure. These variables form the **state vector** used later in the VAR and valuation identity, following the methodology of the paper as closely as possible.

All state variables are:
- constructed at **annual (firm-year) frequency**
- based on **reported, consolidated accounting data**
- computed using **information available at time $ t $** (with lags where required)
- transformed using logs or log(1+x) where appropriate, exactly as in the paper

Observations for which the underlying transformation is not well-defined (e.g. non-positive denominators) are set to missing (`NaN`) rather than forced or winsorized.

***

### 3.1 Valuation States

These variables capture how the firm is priced relative to fundamentals.

- **Book-to-Market**
  $bm_{i,t} = \log\left(\frac{BE_{i,t}}{ME_{i,t}}\right)$

- **Payout Yield**
  $py_{i,t} = \log\left(1 + \frac{PO_{i,t}}{ME_{i,t}}\right)$
  Negative values correspond to net equity issuance rather than cash distributions.

- **Sales Yield**
  $sy_{i,t} = \log\left(\frac{Sales_{i,t}}{ME_{i,t}}\right)$

***

### 3.2 Growth States

These variables capture firm-level growth in size and operations.

- **Book Equity Growth**
  $beg_{i,t} = \log\left(\frac{BE_{i,t}}{BE_{i,t-1}}\right)$

- **Asset Growth**
  $ag_{i,t} = \log\left(\frac{Assets_{i,t}}{Assets_{i,t-1}}\right)$

- **Sales Growth**
  $sg_{i,t} = \log\left(\frac{Sales_{i,t}}{Sales_{i,t-1}}\right)$
  Sales growth is constructed for descriptive purposes and robustness checks, but is not included in the baseline VAR state vector.

***

### 3.3 Profitability States

These variables measure different notions of firm profitability.

- **Clean-Surplus Profitability**
  $csprof_{i,t} = \log\left( 1 + \frac{NI_{i,t} - \Delta BE_{i,t}}{BE_{i,t-1}} \right)$
  This variable plays a central role in the valuation identity, as it directly governs the level of expected future payouts in the duration calculation.

- **Return on Equity**
  $roe_{i,t} = \log\left( 1 + \frac{NI_{i,t}}{\frac{1}{2}(BE_{i,t} + BE_{i,t-1})} \right)$

- **Gross Profitability**
  $gp_{i,t} = \log\left( 1 + \frac{GrossProfit_{i,t}}{\frac{1}{2}(Assets_{i,t} + Assets_{i,t-1})} \right)$

***

### 3.4 Capital Structure State

- **Market Leverage**
  $lev_{i,t} = \frac{Debt_{i,t}}{Debt_{i,t} + ME_{i,t}}$
  Leverage is kept in levels, following the paper.

***

### Remarks

- Log transformations are only applied where economically meaningful (strictly positive inputs).
- Missing values arise naturally from lag construction or accounting edge cases and are retained.
- No standardization, demeaning, or winsorization is applied at this stage.
- This step completes the data preparation required for estimating the VAR in Step 4.

After Step 3, the dataset consists of a firm-year panel with a complete, paper-consistent state vector for each firm.


In [28]:
def build_state_variables(master):
    df = master.copy()

    def safe_log(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index)
        m = x > 0
        out.loc[m] = np.log(x.loc[m])
        return out

    def safe_log1p(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index)
        m = x > -1  # log(1+x) defined iff 1+x>0
        out.loc[m] = np.log1p(x.loc[m])
        return out

    # Valuation
    df["bm"] = safe_log(df["BE"] / df["ME"])
    df["py"] = safe_log1p(df["PO"] / df["ME"])
    df["sy"] = safe_log(df["sales"] / df["ME"])

    # Growth
    df["beg"] = safe_log(df["BE"] / df["BE_lag1"])
    df["ag"]  = safe_log(df["assets"] / df["assets_lag1"])
    df["sg"]  = safe_log(df["sales"] / df["sales_lag1"])

    # Profitability
    df["csprof"] = safe_log1p((df["net_income"] - df["dBE"]) / df["BE_lag1"])
    df["roe"]    = safe_log1p(df["net_income"] / df["avgBE"])
    df["gp"]     = safe_log1p(df["gross_profit"] / df["avgAssets"])

    # Capital structure (robust to pd.NA)
    debt = pd.to_numeric(df["debt"], errors="coerce").to_numpy(dtype=float)
    me   = pd.to_numeric(df["ME"],   errors="coerce").to_numpy(dtype=float)
    denom = debt + me

    lev = np.full(len(df), np.nan, dtype=float)
    mask = denom > 0
    lev[mask] = debt[mask] / denom[mask]
    df["lev"] = lev

    # Clean
    state_vars = ["bm","py","sy","beg","ag","sg","csprof","roe","gp","lev"]
    df[state_vars] = df[state_vars].replace([np.inf, -np.inf], np.nan)

    return df

In [29]:
state_panel = build_state_variables(master)

In [30]:
checks = pd.DataFrame({
    "ME<=0": (master["ME"] <= 0),
    "BE<=0": (master["BE"] <= 0),
    "sales<=0": (master["sales"] <= 0),
    "BE_lag1<=0": (master["BE_lag1"] <= 0),
    "assets_lag1<=0": (master["assets_lag1"] <= 0),
    "sales_lag1<=0": (master["sales_lag1"] <= 0),
    "avgBE<=0": (master["avgBE"] <= 0),
    "avgAssets<=0": (master["avgAssets"] <= 0),
    "1+PO/ME<=0": (1 + master["PO"]/master["ME"] <= 0),
})
checks.mean().sort_values(ascending=False)

BE_lag1<=0        0.004878
avgBE<=0          0.004343
sales_lag1<=0     0.002339
1+PO/ME<=0        0.000869
ME<=0                  0.0
BE<=0                  0.0
sales<=0               0.0
assets_lag1<=0         0.0
avgAssets<=0           0.0
dtype: Float64

## Step 4: Estimation of the Firm-Level State VAR


In this step, we estimate the **dynamic law of motion** for the firm-level state variables constructed in Step 3. The objective is to characterize how valuation, growth, profitability, and capital structure jointly evolve over time and to obtain **long-horizon expectations** required for valuation and duration calculations.

Following the paper, the state dynamics are modeled using a **pooled first-order vector autoregression (VAR(1))**:

$X_{i,t+1} = \mu + \Phi X_{i,t} + \varepsilon_{i,t+1},$

where $ X_{i,t} $ denotes the vector of state variables for firm $ i $ at time $ t $, $ \mu $ is a vector of intercepts, $ \Phi $ is the transition matrix, and $ \varepsilon_{i,t+1} $ is an innovation term.

***

### State Vector Specification

The baseline VAR includes a a carefully selected set of core state variables:

- **Book-to-market ($ bm $)** — valuation anchor  
- **Payout yield ($ py $)** — payout policy  
- **Sales yield ($ sy $)** — scale-adjusted valuation  
- **Asset growth ($ ag $)** — investment dynamics  
- **Return on equity ($ roe $)** — profitability  
- **Market leverage ($ lev $)** — capital structure  
- **Book equity growth ($ beg $)** — long-run firm growth
- **Clean-surplus profitability ($ csprof $)** — payout-generating profitability

These variables are included to ensure internally consistent forecasting of future payouts in the valuation identity.

***

### Estimation Strategy

The VAR is estimated on an **unbalanced firm-year panel** using pooled ordinary least squares. To ensure reliable estimation:

- Observations must have valid state values at both $ t $ and $ t+1 $.
- Firms are required to have a minimum time-series length to enter the estimation sample.
- All state variables are demeaned and standardized using pooled moments for estimation purposes only and are transformed back to levels for forecasting and valuation.

This approach improves numerical stability and ensures comparability across state variables with different units and scales.

***

### Role in the Overall Framework

The baseline VAR includes a carefully selected set of core state variables that
balance parsimony with economic content:

- Book-to-market (bm) — valuation anchor  
- Payout yield (py) — payout policy  
- Sales yield (sy) — scale-adjusted valuation  
- Asset growth (ag) — investment dynamics  
- Return on equity (roe) — profitability  
- Market leverage (lev) — capital structure  
- Book equity growth (beg) — long-run firm growth  
- Clean-surplus profitability (csprof) — payout-generating profitability  

These variables jointly capture the main channels through which firm characteristics
affect expected cash flows and discount rates and ensure internally consistent
forecasting of future payouts in the valuation identity.


In [31]:
# ============================================================
# STEP 5 (PAPER-CLEAN): Extended pooled VAR(1) + forecasting utilities
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ----------------------------
# 5.1 State vector (extended, paper-consistent)
# ----------------------------
# Include beg and csprof because they enter the cashflow identity in Step 6.
var_states = ["bm", "py", "sy", "ag", "roe", "lev", "beg", "csprof"]

# ----------------------------
# 5.2 Build VAR sample (t -> t+1) with strict validity
# ----------------------------
df_var = state_panel.sort_values(["firm_id", "year"]).copy()

# create leads
for v in var_states:
    df_var[f"{v}_lead"] = df_var.groupby("firm_id")[v].shift(-1)

# keep only rows where ALL states exist at t and t+1
req_cols = var_states + [f"{v}_lead" for v in var_states]
df_var = df_var.dropna(subset=req_cols).copy()

# minimum time-series length per firm (strict, paper style)
min_T = 10
firm_counts = df_var.groupby("firm_id").size()
valid_firms = firm_counts[firm_counts >= min_T].index
df_var = df_var[df_var["firm_id"].isin(valid_firms)].copy()

print("STEP 5 VAR sample:")
print(f"  firms: {df_var['firm_id'].nunique()}")
print(f"  obs  : {len(df_var)}")

# ----------------------------
# 5.3 Pooled standardization (on X_t)
# ----------------------------
Z = df_var[var_states].astype(float)
Z_lead = df_var[[f"{v}_lead" for v in var_states]].astype(float)

mu = Z.mean()
sigma = Z.std().replace(0, np.nan)

Zs = (Z - mu) / sigma
Zs_lead = (Z_lead - mu.values) / sigma.values

# ----------------------------
# 5.4 Estimate pooled VAR(1): Z_{t+1} = c + Phi Z_t + u_{t+1}
# ----------------------------
Y = Zs_lead.to_numpy(dtype=float)                # (N x k)
X = sm.add_constant(Zs.to_numpy(dtype=float))    # (N x (k+1))

var_model = sm.OLS(Y, X).fit()

Phi = var_model.params[1:, :].T                 # (k x k)
const = var_model.params[0, :]                  # (k,)

phi_df = pd.DataFrame(Phi, index=var_states, columns=var_states)

# ----------------------------
# 5.5 Stability diagnostics
# ----------------------------
eigvals = np.linalg.eigvals(Phi)
max_eig = float(np.max(np.abs(eigvals)))

print("Extended VAR(1) estimated")
print(f"  Max |eigenvalue|: {max_eig:.4f}")

# ----------------------------
# 5.6 Forecast utilities for Step 6
# ----------------------------
I = np.eye(len(var_states))

# steady state of standardized VAR: zbar = (I - Phi)^{-1} const
zbar = np.linalg.solve(I - Phi, const)

def to_standardized_state(row):
    """Convert a row with raw state variables into standardized z_t."""
    x = row[var_states].to_numpy(dtype=float)
    return (x - mu.to_numpy(dtype=float)) / sigma.to_numpy(dtype=float)

def forecast_states(z0, H):
    """
    Produce E_t[z_{t+h}] for h=1..H in standardized units.
    Returns array shape (H, k).
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    z = z0.copy()
    for h in range(1, H + 1):
        z = const + Phi @ z
        out[h-1, :] = z
    return out

def forecast_states_closedform(z0, H):
    """
    Closed-form version using Phi^h and steady state.
    out[h-1] = Phi^h z0 + (I - Phi^h) zbar
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h-1, :] = Phi_h @ z0 + (I - Phi_h) @ zbar
    return out

# Optional: a quick view of Phi
phi_df


STEP 5 VAR sample:
  firms: 473
  obs  : 9424
Extended VAR(1) estimated
  Max |eigenvalue|: 0.9562


,bm,py,sy,ag,roe,lev,beg,csprof
bm,0.871461,-0.019905,-0.033806,0.016145,-0.012269,0.043667,0.021787,0.004058
py,0.040225,0.197287,0.011326,-0.030817,0.063787,-0.051887,0.007319,0.013297
sy,-0.063913,-0.012577,0.932295,0.025135,-0.019650,0.028658,0.012348,-0.000559
ag,-0.115362,-0.024878,-0.058863,0.028597,0.018270,-0.049610,0.114613,0.037351
roe,-0.268608,0.010556,0.029121,-0.007808,0.367903,0.046769,0.077357,0.088663
lev,-0.017450,-0.015823,-0.018874,0.024748,0.002961,0.944542,0.003432,-0.000533
beg,-0.204968,-0.035350,-0.021237,0.069716,0.003482,0.059595,0.062208,0.039748
csprof,0.005524,0.047801,0.028803,-0.064867,0.269664,-0.043564,0.029176,0.022757


In [32]:
max_eig
phi_df.round(3)

,bm,py,sy,ag,roe,lev,beg,csprof
bm,0.871,-0.020,-0.034,0.016,-0.012,0.044,0.022,0.004
py,0.040,0.197,0.011,-0.031,0.064,-0.052,0.007,0.013
sy,-0.064,-0.013,0.932,0.025,-0.020,0.029,0.012,-0.001
ag,-0.115,-0.025,-0.059,0.029,0.018,-0.050,0.115,0.037
roe,-0.269,0.011,0.029,-0.008,0.368,0.047,0.077,0.089
lev,-0.017,-0.016,-0.019,0.025,0.003,0.945,0.003,-0.001
beg,-0.205,-0.035,-0.021,0.070,0.003,0.060,0.062,0.040
csprof,0.006,0.048,0.029,-0.065,0.270,-0.044,0.029,0.023


## Step 5:  Valuation Identity, Discount Rate, and Equity Duration


### Step 5: Endogenous Discount Rates and Equity Duration

This step constructs firm-year specific **equity discount rates** and **equity duration**
using a valuation-identity approach. The procedure follows a present-value framework in
which the market-to-book ratio equals the discounted value of **expected future payouts
to equityholders**, where expectations are formed using the state dynamics estimated in
Step 4.

***

#### 5.1 Valuation Identity

For each firm $ i $ and year $ t $, the market value of equity relative to book equity
satisfies the identity

$\frac{ME_{i,t}}{BE_{i,t}} = \sum_{h=1}^{\infty} \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r_{i,t} \, h),$

where  
- $ ME_{i,t} $ denotes market equity,  
- $ BE_{i,t} $ denotes book equity,  
- $ PO_{i,t+h} $ denotes payouts to equityholders, and  
- $ r_{i,t} $ is a firm-year specific discount rate.

The discount rate $ r_{i,t} $ is **endogenously determined** such that the valuation
identity holds exactly for each firm-year observation.

***

#### 5.2 Forecasting Expected Payouts

Expected future payouts are constructed from forecasted firm-level state variables
obtained from the VAR estimated in Step 4. Two key state variables govern the payout
process:

- **Book equity growth** $ beg_{i,t} $, which determines the evolution of the scale of
  book equity over time,
- **Clean-surplus profitability** $ csprof_{i,t} $, which governs the share of book
  equity distributed to equityholders.

The payout-to-book ratio at horizon $ h $ is defined as

$\frac{PO_{i,t+h}}{BE_{i,t+h-1}} = \max\left\{ 0,\; \exp(csprof_{i,t+h}) - 1 \right\},$

ensuring that payouts represent **non-negative distributions to equityholders**, while
negative implied values are interpreted as net equity issuance rather than cash payouts.

Expected payouts relative to initial book equity are then given by

$\frac{PO_{i,t+h}}{BE_{i,t}} = \frac{PO_{i,t+h}}{BE_{i,t+h-1}} \cdot \exp\!\left( \sum_{j=1}^{h-1} beg_{i,t+j} \right).$

***

#### 5.3 Finite-Horizon Approximation and Terminal Value

The infinite sum in the valuation identity is approximated using a **finite forecast
horizon $ H $** combined with a conservative terminal value:

$\frac{ME_{i,t}}{BE_{i,t}} = \sum_{h=1}^{H} \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r_{i,t} h) + TV_{i,t}(r_{i,t}).$

The terminal value is specified as a **level perpetuity without growth**, based on the
long-run expected payout ratio,

$TV_{i,t}(r) = \frac{\overline{po}}{\exp(r) - 1} \cdot \exp\!\left( \sum_{j=1}^{H} beg_{i,t+j} \right) \exp(- r H),$

where $ \overline{po} = \max\{0, \exp(csprof_{\infty}) - 1\} $ is the steady-state
payout-to-book ratio implied by the VAR. This conservative terminal specification avoids
explosive valuations and ensures numerically stable discount-rate solutions.

***

#### 5.4 Solving for the Discount Rate

For each firm-year observation, the discount rate $ r_{i,t} $ is obtained as the unique
solution to

$f(r) = \sum_{h=1}^{H} \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r h) + TV_{i,t}(r) - \frac{ME_{i,t}}{BE_{i,t}} = 0.$

The root is computed using a robust bracketing method. Firm-year observations for which
no valid solution exists are excluded from the analysis.

***

#### 5.5 Equity Duration

Equity duration measures the **payout-weighted average maturity** of expected equity cash
flows and is defined as

$Dur_{i,t} = \frac{1}{ME_{i,t}/BE_{i,t}} \left[ \sum_{h=1}^{H} h \cdot \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r_{i,t} h) + Dur^{TV}_{i,t} \right],$

where the terminal contribution equals

$Dur^{TV}_{i,t} = \left( H + \frac{1}{\exp(r_{i,t}) - 1} \right) TV_{i,t}(r_{i,t}).$

This measure captures the effective maturity of equity cash flows, analogous to Macaulay
duration for fixed-income securities.

***

#### 5.6 Sample Construction and Diagnostics

Equity duration and discount rates are computed for firm-year observations with positive
market and book equity, valid VAR forecasts, and a well-defined solution to the valuation
identity. The resulting panel exhibits substantial cross-sectional and time-series
variation. Discount rates and equity duration are negatively correlated, consistent with
asset pricing theory, and the valuation identity is satisfied up to numerical precision.


In [33]:
# ============================================================
# STEP 6 (v3): Stable valuation identity + duration
# Fixes:
#  (A) payouts truncated at 0 (distributions cannot be negative)
#  (B) conservative terminal value (no growth perpetuity)
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import brentq

H = 30
DR_HI = 1.50

k = len(var_states)
I = np.eye(k)

# steady state standardized
try:
    zbar
except NameError:
    zbar = np.linalg.solve(I - Phi, const)

mu_arr = mu.to_numpy(dtype=float)
sig_arr = sigma.to_numpy(dtype=float)

def forecast_states_closedform(z0, H):
    out = np.zeros((H, k), dtype=float)
    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h-1, :] = Phi_h @ z0 + (I - Phi_h) @ zbar
    return out

def unstandardize(z):
    return mu_arr + sig_arr * z

# indices
ix_beg = var_states.index("beg")
ix_cs  = var_states.index("csprof")

# long-run payout ratio level (per BE_{t-1})
xbar_raw = unstandardize(zbar)
csprof_bar = float(xbar_raw[ix_cs])
po_over_be_lag_bar = max(0.0, np.exp(csprof_bar) - 1.0)

def payout_path_to_BE0(beg_path, csprof_path):
    # distributions: max(0, exp(csprof)-1)
    po_over_be_lag = np.maximum(0.0, np.exp(csprof_path) - 1.0)

    # scale by BE_{t+h-1}/BE_t = exp(sum_{j=1}^{h-1} beg_{t+j})
    cum_beg = np.cumsum(beg_path)
    scale = np.exp(np.r_[0.0, cum_beg[:-1]])
    return po_over_be_lag * scale

def terminal_value_BE0(sum_beg_to_H, dr):
    # conservative perpetuity with no growth beyond H
    # TV at H: po_bar / (e^{dr}-1), discounted and scaled by BE growth up to H
    if dr <= 1e-6:
        return np.nan
    annuity = po_over_be_lag_bar / (np.exp(dr) - 1.0)
    return annuity * np.exp(sum_beg_to_H) * np.exp(-dr * H)

def solve_dr(me_be, payout_path, sum_beg_to_H):
    h = np.arange(1, H + 1, dtype=float)

    def f(dr):
        disc = np.exp(-dr * h)
        pv_finite = float(np.sum(payout_path * disc))
        tv = terminal_value_BE0(sum_beg_to_H, dr)
        if not np.isfinite(tv):
            return np.nan
        return (pv_finite + tv) - me_be

    # bracket: low rate must be > 0
    dr_lo = 1e-4
    f_lo = f(dr_lo)
    f_hi = f(DR_HI)

    if not (np.isfinite(f_lo) and np.isfinite(f_hi)):
        raise ValueError("Non-finite at endpoints.")
    if f_lo * f_hi > 0:
        raise ValueError("No sign change.")
    return float(brentq(f, dr_lo, DR_HI, maxiter=200))

def compute_duration(me_be, payout_path, sum_beg_to_H, dr):
    h = np.arange(1, H + 1, dtype=float)
    disc = np.exp(-dr * h)

    pv_finite = float(np.sum(payout_path * disc))
    dur_num_finite = float(np.sum(h * payout_path * disc))

    tv = terminal_value_BE0(sum_beg_to_H, dr)
    if not np.isfinite(tv):
        return np.nan, np.nan

    # terminal is an annuity starting at H (effective maturity ~ H + 1/(e^{dr}-1) )
    # duration contribution of perpetuity: H + 1/(e^{dr}-1)
    dur_tv = (H + (1.0 / (np.exp(dr) - 1.0))) * tv

    pv_total = pv_finite + tv
    dur = (dur_num_finite + dur_tv) / me_be
    pv_check = pv_total - me_be
    return dur, pv_check

# Run
need_cols = ["firm_id", "year", "ME", "BE"] + var_states
df6 = state_panel.dropna(subset=need_cols).copy()
df6 = df6[(df6["ME"] > 0) & (df6["BE"] > 0)].copy()

res = []
for row in df6.itertuples(index=False):
    x0 = np.array([getattr(row, v) for v in var_states], dtype=float)
    z0 = (x0 - mu_arr) / sig_arr

    z_path = forecast_states_closedform(z0, H)
    x_path = unstandardize(z_path)

    beg_path = x_path[:, ix_beg]
    cs_path  = x_path[:, ix_cs]

    payout_path = payout_path_to_BE0(beg_path, cs_path)
    me_be = float(getattr(row, "ME") / getattr(row, "BE"))
    sum_beg_to_H = float(np.sum(beg_path))

    try:
        dr = solve_dr(me_be, payout_path, sum_beg_to_H)
        dur, pv_check = compute_duration(me_be, payout_path, sum_beg_to_H, dr)
        res.append({"firm_id": getattr(row, "firm_id"), "year": int(getattr(row, "year")),
                    "discount_rate": dr, "equity_duration": dur,
                    "pv_check": pv_check, "status": "ok"})
    except Exception:
        res.append({"firm_id": getattr(row, "firm_id"), "year": int(getattr(row, "year")),
                    "discount_rate": np.nan, "equity_duration": np.nan,
                    "pv_check": np.nan, "status": "fail"})

duration_panel_v3 = pd.DataFrame(res)

print("STEP 6 v3 done.")
print("  ok:", (duration_panel_v3["status"]=="ok").sum())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","equity_duration"].describe())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","discount_rate"].describe())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","pv_check"].abs().describe())


STEP 6 v3 done.
  ok: 12231
count    12231.000000
mean        28.141901
std          7.864477
min          1.000000
25%         23.009086
50%         26.425363
75%         31.966660
max        123.326425
Name: equity_duration, dtype: float64
count    12231.000000
mean         0.058264
std          0.023322
min          0.008332
25%          0.048307
50%          0.056992
75%          0.066717
max          1.243347
Name: discount_rate, dtype: float64
count    1.223100e+04
mean     4.012344e-12
std      2.242580e-11
min      0.000000e+00
25%      1.776357e-15
50%      5.129230e-14
75%      2.517542e-12
max      1.746002e-09
Name: pv_check, dtype: float64


In [34]:
ok = duration_panel_v3.query("status=='ok'")
(ok["discount_rate"] > 0.5).mean(), ok["discount_rate"].quantile([0.99, 0.995, 0.999])

(np.float64(0.0002452783909737552),
 0.990    0.095521
 0.995    0.106480
 0.999    0.183351
 Name: discount_rate, dtype: float64)

In [35]:
ok[["equity_duration","discount_rate"]].corr()

,equity_duration,discount_rate
equity_duration,1.000000,-0.554351
discount_rate,-0.554351,1.000000


In [36]:
ok.groupby("firm_id")["equity_duration"].count().describe()


count    934.000000
mean      13.095289
std        9.485506
min        1.000000
25%        4.000000
50%       11.000000
75%       23.000000
max       27.000000
Name: equity_duration, dtype: float64

## Step 6: Build regression-ready Equity Duration panel


In [38]:
# ----------------------------
# 7.1 Keep valid duration observations only
# ----------------------------
dur = duration_panel_v3.query("status == 'ok'").copy()
if "equity_duration" in dur.columns:
    dur = dur.rename(columns={"equity_duration": "Duration_NP"})
if "discount_rate" in dur.columns:
    dur = dur.rename(columns={"discount_rate": "discount_rate_NP"})


# ----------------------------
# 7.2 Merge required firm-year variables from master
# ----------------------------
req_master = ["firm_id", "year", "ME", "BE"]
missing_req = [c for c in req_master if c not in master.columns]
if missing_req:
    raise KeyError(f"master is missing required columns: {missing_req}")

dur = dur.merge(
    master[req_master].copy(),
    on=["firm_id", "year"],
    how="left"
)

# ----------------------------
# 7.3 Bring in optional controls
#     Priority:
#       (1) from master if available
#       (2) else from state_panel if available
# ----------------------------
optional_controls = ["bm", "ag", "roe", "lev"]

# (1) from master if present
opt_from_master = [c for c in optional_controls if c in master.columns]
if opt_from_master:
    dur = dur.merge(
        master[["firm_id", "year"] + opt_from_master].copy(),
        on=["firm_id", "year"],
        how="left"
    )

# (2) from state_panel if still missing and state_panel exists
still_missing = [c for c in optional_controls if c not in dur.columns]
if still_missing and "state_panel" in globals():
    opt_from_state = [c for c in still_missing if c in state_panel.columns]
    if opt_from_state:
        dur = dur.merge(
            state_panel[["firm_id", "year"] + opt_from_state].copy(),
            on=["firm_id", "year"],
            how="left"
        )

# If bm is missing entirely, construct a simple bm from BE/ME (levels)
if "bm" not in dur.columns:
    dur["bm"] = dur["BE"] / dur["ME"]

# ----------------------------
# 7.4 Final column selection (keep what exists)
# ----------------------------
base_cols = ["firm_id", "year", "Duration_NP", "discount_rate_NP", "ME", "BE", "bm"]
for c in ["ag", "roe", "lev"]:
    if c in dur.columns:
        base_cols.append(c)

dur = dur[base_cols].copy()

# ----------------------------
# 7.5 Basic cleaning
# ----------------------------
dur = dur.dropna(subset=["Duration_NP", "discount_rate_NP", "ME", "BE", "bm"])
dur = dur[(dur["Duration_NP"] > 0) & (dur["ME"] > 0) & (dur["BE"] > 0)]

# ----------------------------
# 7.6 Optional winsorized duration for regressions
# ----------------------------
dur["Duration_NP_w"] = dur["Duration_NP"].clip(
    lower=dur["Duration_NP"].quantile(0.01),
    upper=dur["Duration_NP"].quantile(0.99),
)

# ----------------------------
# 7.7 Diagnostics
# ----------------------------
print("Firm-level coverage:")
print(dur.groupby("firm_id")["Duration_NP"].count().describe())

print("Year-level coverage:")
print(dur.groupby("year")["Duration_NP"].count())

print("Duration summary:")
print(dur["Duration_NP"].describe())

print("Discount rate summary:")
print(dur["discount_rate_NP"].describe())

# ----------------------------
# 7.8 Merge durations_panel into equity_duration_panel
# (also bring in BETA_5Y)
# ----------------------------
dur_path = DATA_DIR / "durations_panel.parquet"
if dur_path.exists():
    dur_extra = pd.read_parquet(dur_path).copy()

    # harmonize year column name
    if "YEAR" in dur_extra.columns and "year" not in dur_extra.columns:
        dur_extra = dur_extra.rename(columns={"YEAR": "year"})
    elif "AsOfYear" in dur_extra.columns and "year" not in dur_extra.columns:
        dur_extra = dur_extra.rename(columns={"AsOfYear": "year"})

    dur_extra["year"] = pd.to_numeric(dur_extra["year"], errors="coerce").astype("Int64")

    if "firm_id" in dur_extra.columns:
        key_cols = {"firm_id", "year", "YEAR", "AsOfYear"}

        cols_to_add = [
            c for c in dur_extra.columns
            if c not in key_cols and (c.startswith("Duration_") or c.startswith("D_") or c == "BETA_5Y")
        ]
        cols_to_add = [c for c in cols_to_add if c not in dur.columns]

        if cols_to_add:
            dur = dur.merge(
                dur_extra[["firm_id", "year"] + cols_to_add].drop_duplicates(["firm_id", "year"]),
                on=["firm_id", "year"],
                how="left",
            )
            print("Added columns from durations_panel:", cols_to_add)
        else:
            print("No new columns added from durations_panel.")
    else:
        print("durations_panel has no firm_id; skipped merge to keep firm_id-only keys.")
else:
    print(f"durations_panel not found at: {dur_path}")

# ----------------------------
# 7.9 Add membership metadata (Company, HQ, Sectors) — EURO500 net-payout
# ----------------------------
if "euro500_netpayout" in globals() and euro500_netpayout is not None and not euro500_netpayout.empty:
    mem = euro500_netpayout.copy()
else:
    mem_path = DATA_DIR / "euro500_netpayout.parquet"
    if mem_path.exists():
        mem = pd.read_parquet(mem_path)
    else:
        mem = None

if mem is not None:
    mem = mem.copy()

    if "firm_id" not in mem.columns:
        raise KeyError("euro500_netpayout metadata must contain firm_id")
    mem["firm_id"] = mem["firm_id"].astype(str).str.strip()

    rename_map = {}
    if "CompanyName" not in mem.columns and "name" in mem.columns:
        rename_map["name"] = "CompanyName"
    if "CountryHQ" not in mem.columns and "hq_country" in mem.columns:
        rename_map["hq_country"] = "CountryHQ"
    if "CountryHQCode" not in mem.columns and "hq_code" in mem.columns:
        rename_map["hq_code"] = "CountryHQCode"
    if "EconomicSector" not in mem.columns and "trbc_sector" in mem.columns:
        rename_map["trbc_sector"] = "EconomicSector"
    if "EconomicSectorCode" not in mem.columns and "trbc_sector_code" in mem.columns:
        rename_map["trbc_sector_code"] = "EconomicSectorCode"

    if rename_map:
        mem = mem.rename(columns=rename_map)

    meta_cols = [c for c in [
        "firm_id",
        "CompanyName",
        "CountryHQ",
        "CountryHQCode",
        "EconomicSector",
        "EconomicSectorCode",
        "mcap_eur",
    ] if c in mem.columns]

    if "date" in mem.columns:
        mem["date"] = pd.to_datetime(mem["date"], errors="coerce")
        mem = mem.sort_values(["firm_id", "date"])
        mem_meta = (
            mem[meta_cols + ["date"]]
            .groupby("firm_id", as_index=False)
            .last()
        )
        mem_meta = mem_meta.drop(columns=["date"], errors="ignore")
    else:
        mem_meta = mem[meta_cols].drop_duplicates("firm_id")

    dur = dur.merge(mem_meta, on="firm_id", how="left")

    print("Added EURO500 net-payout metadata columns:", [c for c in meta_cols if c != "firm_id"])
else:
    print("euro500_netpayout parquet not found; skipping metadata merge.")

# ----------------------------
# 7.10 Column order
# ----------------------------
preferred = [
    "firm_id", "year",
    "CompanyName", "CountryHQ", "EconomicSector", "BusinessSector",
    "Duration_NP", "Duration_NP_w", "discount_rate_NP",
    "ME", "BE", "bm", "ag", "roe", "lev",
]

existing_pref = [c for c in preferred if c in dur.columns]
remaining = [c for c in dur.columns if c not in existing_pref]

# Keep any additional duration columns from durations_panel close to main duration block
extra_dur = [c for c in remaining if c.startswith("Duration_") or c.startswith("D_")]
remaining = [c for c in remaining if c not in extra_dur]

dur = dur[existing_pref + extra_dur + remaining]

# ----------------------------
# 7.10 Save
# ----------------------------
save_parquet(dur, "equity_duration_panel")


Firm-level coverage:
count    934.000000
mean      13.095289
std        9.485506
min        1.000000
25%        4.000000
50%       11.000000
75%       23.000000
max       27.000000
Name: Duration_NP, dtype: float64
Year-level coverage:
year
1999    339
2000    375
2001    377
2002    426
2003    437
2004    452
2005    449
2006    442
2007    433
2008    472
2009    484
2010    482
2011    468
2012    456
2013    464
2014    471
2015    473
2016    476
2017    468
2018    467
2019    474
2020    476
2021    471
2022    484
2023    468
2024    477
2025    470
Name: Duration_NP, dtype: int64
Duration summary:
count    12231.000000
mean        28.141901
std          7.864477
min          1.000000
25%         23.009086
50%         26.425363
75%         31.966660
max        123.326425
Name: Duration_NP, dtype: float64
Discount rate summary:
count    12231.000000
mean         0.058264
std          0.023322
min          0.008332
25%          0.048307
50%          0.056992
75%          0.06671